# optimizer-repr-string — ex1: give a hand-rolled SGD a debug-friendly __repr__

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-repr-string`. Running the final beacon cell reports progress against the `Optimizer: __repr__ string` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: __repr__ string` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-repr-string`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-repr-string"
DD_SUBTOPIC = "Optimizer: __repr__ string"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Optimizer: `__repr__` string — quick refresher

Implementing `__repr__` gives your optimizer a debug-friendly string in tracebacks, REPL printouts, and log lines. The convention is `ClassName(arg1=value1, arg2=value2)` — looks like the constructor that would re-create it:

```python
class SGD:
    def __init__(self, params, lr, momentum=0.0):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum

    def __repr__(self):
        return f'SGD(lr={self.lr}, momentum={self.momentum})'

>>> opt = SGD(model.parameters(), lr=1e-3, momentum=0.9)
>>> opt
SGD(lr=0.001, momentum=0.9)
```

**`__repr__` vs `__str__`.** `__repr__` is for developers — unambiguous, ideally `eval`-able. `__str__` is for end users — pretty-printed. When `__str__` is not defined, `str(x)` falls back to `__repr__`. For optimizers, only `__repr__` is needed.

**Why include hyperparameters, not the full param list.** The param tensors are bulky and uninformative in a debug print. Hyperparameters (`lr`, `momentum`, `weight_decay`) are what you actually want to see when checking that the optimizer was constructed correctly.

**PyTorch's own optimizer repr.** `torch.optim.SGD` prints a multi-line repr showing each param group's hparams. The pattern is the same — show config, hide bulk data.

**Use f-strings, not `%` formatting.** Modern, readable, and evaluates `self.lr` etc. inline.

### Exercise 1 — give a hand-rolled SGD a debug-friendly __repr__

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `__repr__` to a custom optimizer class so its string form lists hyperparameters in a ClassName(arg=value, ...) shape and `repr(opt)` returns the same.
> Keywords: repr, dunder, optimizer, debug
> ```

**KCs targeted:** `repr-returns-constructor-like-string`, `repr-includes-hparams-excludes-bulk`

Complete the `SGD` class below by implementing `__repr__` so it returns the string:

    'SGD(lr=<lr>, momentum=<momentum>)'

where `<lr>` and `<momentum>` are the current attribute values, formatted using their default `repr()` (i.e. f-string `{self.lr}` and `{self.momentum}`).

Inputs / state:
- `self.lr`: float
- `self.momentum`: float
- `self.params`: list (the actual parameter tensors — do NOT include them in the repr)

Constraints:
- The output is a single line.
- The format is exactly `SGD(lr=<lr>, momentum=<momentum>)` with one space after the comma (matches Python's default f-string formatting).
- `repr(opt)` and `str(opt)` must both produce the same string (when `__str__` is not defined, it falls back to `__repr__`).

In [ ]:
class SGD:
    def __init__(self, params, lr, momentum=0.0):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum

    def __repr__(self):
        raise NotImplementedError()


def _test_ex1():
    # === Basic case ===
    params = [t.zeros(10), t.zeros(20)]
    opt = SGD(params, lr=0.001, momentum=0.9)
    r = repr(opt)
    assert isinstance(r, str), f'__repr__ must return str, got {type(r).__name__}'
    assert r == 'SGD(lr=0.001, momentum=0.9)', f'expected exact format, got {r!r}'

    # === str() falls back to __repr__ when __str__ is undefined ===
    assert str(opt) == repr(opt), 'str(opt) and repr(opt) must agree'

    # === Different values ===
    opt2 = SGD([t.zeros(1)], lr=0.1, momentum=0.0)
    assert repr(opt2) == 'SGD(lr=0.1, momentum=0.0)', f'got {repr(opt2)!r}'

    # === Repr does NOT include the params list (would be huge) ===
    big_opt = SGD([t.randn(10_000)], lr=1e-4, momentum=0.99)
    r = repr(big_opt)
    assert 'tensor' not in r.lower(), (
        f'repr should NOT include the param tensors, got {r!r}'
    )
    assert len(r) < 100, f'repr is way too long, got {len(r)} chars: {r!r}'

    # === Class name is in the repr (so tracebacks identify the type) ===
    assert r.startswith('SGD('), f'repr should start with class name, got {r!r}'

    # === Mutating an attribute changes the repr live ===
    opt = SGD([t.zeros(3)], lr=1e-3, momentum=0.5)
    r1 = repr(opt)
    opt.lr = 1e-5
    r2 = repr(opt)
    assert r1 != r2, 'repr should reflect current attribute values'
    assert 'lr=1e-05' in r2, f'expected updated lr to show, got {r2!r}'

    # === Both hparams appear ===
    opt = SGD([t.zeros(1)], lr=0.5, momentum=0.5)
    r = repr(opt)
    assert 'lr=' in r and 'momentum=' in r, f'both hparams must appear, got {r!r}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
class SGD:
    def __init__(self, params, lr, momentum=0.0):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum

    def __repr__(self):
        return f'SGD(lr={self.lr}, momentum={self.momentum})'
```

**Constructor-mirror format.** `SGD(lr=0.001, momentum=0.9)` reads like the call that would re-create the object — ideal for debugging and logging. PyTorch's own `torch.optim.Optimizer.__repr__` follows the same pattern (just multi-line because of param groups).

**Hparams in, params out.** The `self.params` list could be megabytes for a real model. Hyperparameters are what you actually want to see in a print statement; weights belong in `state_dict`, not `repr`.

**`__str__` falls back to `__repr__`.** When `__str__` isn't defined, `str(x)` returns `repr(x)`. That's why the test asserts `str(opt) == repr(opt)` — you got both for free.

**f-string formatting matches Python's defaults.** `f'{self.lr}'` formats a float as Python's default (e.g. `0.001`, `1e-05`). Don't try to special-case scientific vs decimal; let Python decide.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()